In [20]:
import numpy as np
import pandas as pd

true_rates = np.array([0.02, 0.04, 0.035, 0.06])
best_arm = int(np.argmax(true_rates))
best_rate = true_rates[best_arm]

def pull_arm(arm, rng):
    return int(rng.random() < true_rates[arm])

In [21]:
pd.DataFrame({
    'true_success_rate': true_rates,
    'is_best': np.arange(len(true_rates)) == best_arm,
})

,true_success_rate,is_best
0,0.020,False
1,0.040,False
2,0.035,False
3,0.060,True


## Задача 1. Посчитать cumulative regret

Напишите функцию `cumulative_regret`, которая принимает массив выбранных ручек `chosen_arms` и возвращает массив накопленного regret.

Формула instant regret:

$$
\mu^* - \mu_{A_t}.
$$

Формула cumulative regret:

$$
\operatorname{Regret}(T)
=
\sum_{t=1}^{T}
\left(\mu^* - \mu_{A_t}\right).
$$

In [22]:
def cumulative_regret(chosen_arms):
    chosen_rates = true_rates[np.asarray(chosen_arms)]
    instant_regret = best_rate - chosen_rates
    return np.cumsum(instant_regret)

In [23]:
test_arms = np.array([0, 3, 1, 2])
reg = cumulative_regret(test_arms)
expected = np.cumsum(best_rate - true_rates[test_arms])
assert isinstance(reg, np.ndarray)
assert np.allclose(reg, expected)
assert len(reg) == len(test_arms)
print('Задача 1: OK')

Задача 1: OK


## Задача 2. Реализовать выбор действия для epsilon-greedy

Напишите функцию `select_epsilon_greedy_arm`.

Она должна:

- с вероятностью `epsilon` выбирать случайную ручку;
- с вероятностью `1 - epsilon` выбирать ручку с максимальной текущей оценкой `estimated_rates`.

Если максимумов несколько, можно брать первый `np.argmax`.

In [24]:
def select_epsilon_greedy_arm(estimated_rates, epsilon, rng):
    n_arms = len(estimated_rates)

    if rng.random() < epsilon:
        arm = rng.integers(0, n_arms)
    else:
        arm = int(np.argmax(estimated_rates))
    return arm
        

In [25]:
rng = np.random.default_rng(123)
estimated = np.array([0.1, 0.3, 0.2])

for _ in range(20):
    assert select_epsilon_greedy_arm(estimated, 0.0, rng) == 1

arms = [select_epsilon_greedy_arm(estimated, 1.0, rng) for _ in range(100)]
assert set(arms).issubset({0, 1, 2})
assert len(set(arms)) > 1
print('Задача 2: OK')

Задача 2: OK


## Задача 3. Реализовать полный epsilon-greedy эксперимент

Напишите функцию `run_epsilon_greedy`.

Требования:

- первые `n_arms` шагов попробовать каждую ручку по одному разу;
- дальше использовать `select_epsilon_greedy_arm`;
- хранить выбранные ручки и rewards;
- вернуть `DataFrame` с колонками `t`, `arm`, `reward`.

In [26]:
def run_epsilon_greedy(n_steps, epsilon, rng):
    n_arms = len(true_rates)
    counts = np.zeros(n_arms, dtype=int)
    reward_sums = np.zeros(n_arms, dtype=float)
    chosen_arms = []
    rewards = []

    # YOUR CODE HERE
    for t in range(n_steps):
        if t < n_arms:
            arm = t
        else:
            select_epsilon_greedy_arm(estimated, epsilon, rng)
        reward = pull_arm(arm, rng)
        counts[arm] += 1
        reward_sums[arm] += reward

        chosen_arms.append(arm)
        rewards.append(reward)

    return pd.DataFrame({
            't': np.arange(1, n_steps + 1),
            'arm': chosen_arms,
            'reward': rewards
        })

In [27]:
hist = run_epsilon_greedy(1000, epsilon=0.1, rng=np.random.default_rng(1))
assert isinstance(hist, pd.DataFrame)
assert list(hist.columns) == ['t', 'arm', 'reward']
assert len(hist) == 1000
assert set(hist['reward'].unique()).issubset({0, 1})
assert set(hist['arm'].unique()).issubset(set(range(len(true_rates))))
assert hist['t'].iloc[0] == 1
assert hist['t'].iloc[-1] == 1000
print('Задача 3: OK')

Задача 3: OK


## Задача 4. Реализовать Thompson Sampling

Для каждой ручки храним параметры Beta-распределения:

$$
\theta_a \sim \operatorname{Beta}(\alpha_a, \beta_a).
$$

На каждом шаге:

1. сэмплируем значение из Beta-распределения для каждой ручки;
2. выбираем ручку с максимальным сэмплом;
3. получаем reward;
4. обновляем параметры:

$$
\alpha_a \leftarrow \alpha_a + r,
\qquad
\beta_a \leftarrow \beta_a + 1-r.
$$

In [30]:
def run_thompson_sampling(n_steps, rng):
    n_arms = len(true_rates)
    alpha = np.ones(n_arms)
    beta = np.ones(n_arms)
    chosen_arms = []
    rewards = []

    for t in range(n_steps):
        sampled_rates = rng.beta(alpha, beta)
        arm = int(np.argmax(sampled_rates))

        reward = pull_arm(arm, rng)

        alpha[arm] += reward
        beta[arm] += 1 - reward

        chosen_arms.append(arm)
        rewards.append(reward)

    # YOUR CODE HERE
    return pd.DataFrame({
        't': np.arange(1, n_steps + 1),
        'arm': chosen_arms,
        'reward': rewards
    })

In [31]:
ts_hist = run_thompson_sampling(1000, rng=np.random.default_rng(2))
assert isinstance(ts_hist, pd.DataFrame)
assert list(ts_hist.columns) == ['t', 'arm', 'reward']
assert len(ts_hist) == 1000
assert set(ts_hist['reward'].unique()).issubset({0, 1})
assert set(ts_hist['arm'].unique()).issubset(set(range(len(true_rates))))
print('Задача 4: OK')

Задача 4: OK


## Задача 5. Сравнить алгоритмы

Запустите `epsilon-greedy` и `Thompson Sampling` на `5000` шагов.

Сравните:

- total reward;
- final cumulative regret;
- долю выборов лучшей ручки.

Сохраните итоговую таблицу в переменную `summary`.

In [ ]:
n_steps = 5000
best_arm = np.argmax(true_rates)
max_rate = np.max(true_rates)

rng_eps = np.random.default_rng(42)
rng_ts = np.random.default_rng(42)

hist_eps = run_epsilon_greedy(n_steps=n_steps, epsilon=0.1, rng=rng_eps)
hist_ts = run_thompson_sampling(n_steps=n_steps, rng=rng_ts)

def calculate_metrics(hist):
    total_reward = hist['reward'].sum()
    
    chosen_arm_rates = np.array(true_rates)[hist['arm']]
    final_regret = np.sum(max_rate - chosen_arm_rates)
    
    best_arm_share = (hist['arm'] == best_arm).mean()
    
    return {
        'total_reward': total_reward,
        'final_regret': final_regret,
        'best_arm_share': best_arm_share
    }

summary = pd.DataFrame({
    'Epsilon-Greedy': calculate_metrics(hist_eps),
    'Thompson Sampling': calculate_metrics(hist_ts)
}).T

display(summary)

,total_reward,final_regret,best_arm_share
Epsilon-Greedy,304.0,0.085,0.9994
Thompson Sampling,258.0,36.195,0.6716


In [35]:
assert 'summary' in globals()
assert isinstance(summary, pd.DataFrame)
required_cols = {'total_reward', 'final_regret', 'best_arm_share'}
assert required_cols.issubset(set(summary.columns))
assert len(summary) >= 2
assert np.all(summary['best_arm_share'].between(0, 1))
print('Задача 5: OK')

Задача 5: OK
